# 02 · Kaplan-Meier survival curves

Run each cell with **Shift+Enter**. Cells under **Your turn** contain `...` placeholders; replace them, run, then run the check beneath. Checks print `PASS` or `NOT YET` and never stop the notebook. Worked answers are in the matching `_solutions` notebook.

The checks compare your results with the **truth in `python/simulate.py`**, so they keep working if you change the simulator's settings.

| R `survival` | Python `lifelines` |
| --- | --- |
| `survfit(Surv(duration, lapsed) ~ 1)` | `KaplanMeierFitter().fit(duration, event_observed=lapsed)` |
| `summary(fit, times = c(12, 24))` | `kmf.predict([12, 24])` |
| `survdiff(Surv(...) ~ group)` | `multivariate_logrank_test(duration, group, lapsed)` |

Members who are still active at the end of observation are **censored**: we know they lasted at least `duration` months, not when they will leave. Kaplan-Meier handles that correctly; simple proportions do not.

In [ ]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [ ]:
import sys
sys.path.insert(0, "../python")

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulate as sim
importlib.reload(sim)

members = pd.read_parquet("../data/members.parquet")
print(f"{len(members):,} members, {members['lapsed'].mean():.1%} lapsed")

from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

## 1. One curve for the whole book

### Your turn 1.1

Fit `kmf` on `duration` with `event_observed=lapsed`, plot it, and set `s12` and `s24`: the estimated share still members at 12 and 24 months (`kmf.predict(12)`).

In [ ]:
kmf = ...
s12 = ...
s24 = ...

In [ ]:
check("1.1 survival falls over time", lambda: 0 < s24 < s12 < 1)

### Your turn 1.2: why censoring matters

A tempting shortcut is "the share of members who have not lapsed by month 24". Compute `naive_s24` as `1 -` the share of **all** members who lapsed at or before month 24. It counts recent joiners, who have not yet had 24 months to lapse, as survivors, so it overstates survival.

In [ ]:
naive_s24 = ...

In [ ]:
check("1.2 the naive estimate overstates survival", lambda: naive_s24 > s24 + 0.02)

## 2. Curves by segment

### Your turn 2.1

Fit one curve per `product_tier`, plot them on one chart, and build `s24_by_tier`: a dict of tier to survival at 24 months.

Hint: loop over `members.groupby("product_tier")`, fit a new `KaplanMeierFitter(label=tier)` for each, and pass `ax=ax` to `plot_survival_function` so the curves share a chart.

In [ ]:
s24_by_tier = {}
...

In [ ]:
check("2.1 top tier retains best", lambda: s24_by_tier["top"] > s24_by_tier["basic"])

### Your turn 2.2

Test whether the tiers' curves differ with a log-rank test. Set `p_tier` to the p-value (`multivariate_logrank_test(durations, groups, event_observed).p_value`).

In [ ]:
p_tier = ...

In [ ]:
check("2.2 tiers clearly differ", lambda: p_tier < 0.001)

## Wrap-up

Look at the curves around months 12, 24 and 36: each anniversary shows up as a sharper step down. Commit from VS Code: `Kaplan-Meier curves by tier`.